# Synthetic Enhanced Dataset Pipeline

This notebook runs the standalone dataset generator pipeline.
It downloads pre-processed CSV datasets, replaces malicious IPs with realistic IPs from Threat Intelligence feeds, and standardizes public benign IPs.

1. **Setup** — ML stack.
2. **Configure** target days and run parameters
3. **Run** pipeline


## Configuration


In [4]:
!git clone https://github.com/devedale/synth-cic-ids-2018

Cloning into 'synth-cic-ids-2018'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 61 (delta 20), reused 57 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 41.75 KiB | 5.96 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [5]:
%cd synth-cic-ids-2018/
!pip install -r requirements.txt

/content/synth-cic-ids-2018
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00


In [7]:

import sys
import pandas as pd
from pathlib import Path

# Override settings parameters dynamically if needed
import configs.settings as _s

DAYS = [
    'Thursday-01-03-2018',
    'Friday-02-03-2018',
]
FORCE_RERUN = False
SAMPLE_SIZE = 500000

_s.DAYS = DAYS
_s.SAMPLE_SIZE = SAMPLE_SIZE

print('Running for days:', DAYS)


Running for days: ['Thursday-01-03-2018', 'Friday-02-03-2018']


## Run Pipeline


In [ ]:
from core.ingestion import Ingestion
from core.preprocessing import preprocess, clean_temp

PIPELINE_ROOT = Path('.').resolve()

ing = Ingestion(base_dir=PIPELINE_ROOT)
raw = ing.run(days=DAYS, force_rerun=FORCE_RERUN)
raw_df = raw['dataframe']

preproc_df = preprocess(
    raw_df,
    sample_size=SAMPLE_SIZE,
    cache=_s.CACHE_ENABLED,
    cache_dir=_s.CACHE_DIR,
)

clean_temp(PIPELINE_ROOT)

print('\nPipeline Terminated!')
print('Total records:', len(raw_df))


[ingestion] Aggregating malicious IPs from 1 enabled feeds...
[ingestion] Fetching malicious: https://raw.githubusercontent.com/borestad/blocklist-abuseipdb/main/abuseipdb-s100-30d.ipv4
[ingestion] Loaded 142853 unique malicious IPs.
[ingestion] Aggregating benign IPs from 4 enabled feeds...
[ingestion] Fetching benign: https://raw.githubusercontent.com/borestad/iplists/refs/heads/main/google/googlebot.ipv4
[ingestion] Fetching benign: https://raw.githubusercontent.com/borestad/iplists/refs/heads/main/bing/bingbot.ipv4
[ingestion] Fetching benign: https://raw.githubusercontent.com/borestad/iplists/refs/heads/main/apple/apple.ipv4
[ingestion] Fetching benign: https://raw.githubusercontent.com/borestad/iplists/refs/heads/main/email/office365-exchange-smtp.ipv4
[ingestion] Loaded 106 unique benign IPs.
[ingestion] Processing day: Thursday-01-03-2018
[ingestion] Downloading s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
[inge

## Results Verification


In [10]:
if not raw_df.empty:
    print('=== Label distribution ===')
    print(raw_df['Label'].value_counts().to_string())

    if 'Src IP' in raw_df.columns:
        print('\n=== Sample from IPs (Modified) ===')
        display(raw_df[['Src IP', 'Dst IP', 'Label']].sample(min(10, len(raw_df))))
else:
    print('No results fetched.')


=== Label distribution ===
Label
Benign           1000421
Bot               286191
Infilteration      93063
Label                 25

=== Sample from IPs (Modified) ===


,Src IP,Dst IP,Label
616167,10.246.249.108,172.17.81.189,Benign
517319,172.25.158.82,192.168.180.114,Benign
1132171,136.107.128.191,10.19.88.179,Bot
253592,161.35.167.111,172.30.35.6,Infilteration
480591,172.25.86.186,172.23.226.136,Benign
529133,10.15.129.197,192.168.180.133,Benign
344917,10.104.7.167,192.168.108.158,Benign
805889,192.168.19.245,172.23.37.19,Benign
487187,172.24.179.111,10.135.96.7,Benign
459066,192.168.17.36,192.168.184.88,Benign


Check Synthetic dataset https://www.abuseipdb.com/